# Telco Customer Churn Prediction

This project aims to predict customer churn using python with the pandas and sklearn libraries. Logistic regression and random forest classifier were the models used in this project. SMOTE was applied to address the imbalanced dataset.

Link to dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn/discussion?sort=hotness

## Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
df = pd.read_csv('/Users/vivianchung/Desktop/projects/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Check for null values
df.isnull().sum()

In [ ]:
# Check for duplicates
df.duplicated().sum()

In [ ]:
# Calculate the percentage distribution of 'Churn' values
round(df['Churn'].value_counts()/df.shape[0]*100)

### Class imbalance

About 73% of customers did not churn and about 27% of customers did churn. The dataset is imbalanced because a larger proportion of customers did not churn compared to those that did.

### Actions to consider

SMOTE or Synthetic Minority Over-sampling Technique will be applied to the dataset to address the class imbalance issue.


In [ ]:
#ok = [488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]

## 1. Change 'TotalCharges' from object to numeric

In [ ]:
# Change 'TotalCharges' from object to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.info()

## 2. Remove NaN in 'TotalCharges' column

In [ ]:
df['TotalCharges'].isnull().sum()

In [ ]:
df[df['TotalCharges'].isnull()]

In [ ]:
df['TotalCharges'].isnull().sum() / df.shape[0] * 100

0.156% of the data is missing in the 'TotalCharges' column. Since this value is small, I decided to remove the rows with missing values. 

In [ ]:
# Drop rows with missing values in 'TotalCharges' column
df.dropna(subset=['TotalCharges'], inplace=True)

In [ ]:
df['TotalCharges'].isnull().sum()

In [ ]:
df.head()

## 3. Encode categorical variables

In [ ]:
# Drop 'customerID'
df.drop('customerID', axis=1, inplace=True)

# Replace 'No phone service' and 'No internet service' to 'No' for the following columns:
# MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies
df['MultipleLines'].replace('No phone service', 'No', inplace=True)
df.replace('No internet service', 'No', inplace=True)

# Gender, Partner, Dependents, PhoneService, PaperlessBilling, Churn --> Binary
df['gender'].replace({'Male':0, 'Female':1}, inplace=True)
df.replace({'No':0, 'Yes':1}, inplace=True)

# SeniorCitizen --> NA

# tenure, TotalCharges, MonthlyCharges --> NA

# InternetService, Contract, PaymentMethod --> One Hot
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'])
df.replace({False:0, True:1}, inplace=True)

In [ ]:
df.head()

# Logistic Regression 

In [ ]:
# Define features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
# Train the model
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_resampled, y_resampled)

In [ ]:
# Make predictions
y_pred = lr_model.predict(X_test)

In [ ]:
# Evaluate the model
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('Accuracy Score:', accuracy_score(y_test, y_pred))

# Random Forest Classifier

In [ ]:
# Train the model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_resampled, y_resampled)

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)

In [ ]:
# Evaluate the model
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('Accuracy Score:', accuracy_score(y_test, y_pred))

## Logistic regression
Precision for Class 0 (non-churn) is high, meaning the model is good at predicting non-churn cases accurately. Precision for Class 1 (churn) is lower, indicating that when the model predicts churn, it is less accurate.

Recall for Class 1 is high, suggesting that the model is good at identifying most of the actual churn cases. Recall for Class 0 is lower, meaning the model misses some non-churn cases.

The model correctly predicts the outcome 74% of the time. 

#### Strengths:
The model is fairly good at detecting churn cases (high recall for Class 1).

It has a high precision for non-churn cases, indicating reliability in predicting non-churn when it makes a prediction.

#### Weaknesses:
The model struggles with precision for churn cases, meaning it may generate a higher number of false positives (predicting churn when it?s not actually churn).

The F1-score for churn is lower, which indicates a trade-off between precision and recall that could be improved.

## Random Forest
Precision is higher for Class 0 (non-churn), showing the model is good at predicting non-churn cases accurately. For Class 1 (churn), precision is lower, indicating that the model is less reliable when predicting churn.

Recall for both classes is equal, reflecting the model's ability to identify churn and non-churn cases. The recall is lower for Class 1, meaning it misses some actual churn cases.

The model correctly predicts the outcome 78% of the time. This accuracy is better than the Logistic Regression model but still shows room for improvement, particularly in the prediction of churn cases.

#### Strengths
Better accuracy compared to Logistic Regression.

Balanced performance across classes with an overall higher precision and recall.

#### Weaknesses: 
Precision and recall for churn cases (Class 1) are lower, indicating that while the model detects some churn cases, it misses or incorrectly labels others.


# Conclusion
Both models provide valuable insights, but the Random Forest model's higher accuracy and better balance across classes may make it more suitable for scenarios where better overall performance is desired.